# CNDE NDT Document QA — Grounded RAG with Automated Verification
Run the sections **in order** on a fresh Colab runtime with a **T4 GPU** (`Runtime > Change runtime type`).
Reusable logic lives in `src/`; this notebook only wires it together.

## 1. Environment setup

In [1]:
import sys, platform, shutil, subprocess
print("Python", sys.version.split()[0], "|", platform.platform())
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)
else:
    print("NO GPU on this runtime -> switch to a T4 GPU runtime before section 5.")

Python 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07



## 2. Dependency installation
Uses the code synced by the Colab folder-sync extension (`/content/cnde`) if present; otherwise clones the public repo. torch is preinstalled on Colab and is not reinstalled.

In [2]:
import os
REPO_URL = "https://github.com/AbhijeetKumar1505/CNDE.git"
BRANCH = "main"                # or a feature branch, e.g. "feature/llm-verification"
SYNCED_DIR = "/content/cnde"    # local folder synced by the Colab sync extension
CLONE_DIR = "/content/CNDE"

if os.path.isdir(os.path.join(SYNCED_DIR, "src")):
    REPO_DIR = SYNCED_DIR
    print("Using synced working copy:", REPO_DIR)
else:
    REPO_DIR = CLONE_DIR
    if not os.path.isdir(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}
    !git -C {REPO_DIR} log --oneline -3
%cd {REPO_DIR}
%pip install -q -r requirements.txt

Cloning into '/content/CNDE'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 116 (delta 60), reused 73 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 124.11 KiB | 1.01 MiB/s, done.
Resolving deltas: 100% (60/60), done.
3b83ea7 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #8 from AbhijeetKumar1505/feature/llm-verification
c07138d (origin/feature/llm-verification) Notebook: reload app.py and reload modules in dependency order
4667ed5 Merge pull request #7 from AbhijeetKumar1505/feature/llm-verification
/content/CNDE


## 3. Google Drive mounting
Persistent storage: `MyDrive/CNDE_Hackathon/`. If mounting is unavailable (e.g. VS Code Colab kernel), `Config` falls back to `<repo>/workspace` on the runtime, which does not persist across runtimes.

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:  # report, don't hide: storage falls back to the runtime-local workspace
    print(f"Drive NOT mounted ({type(e).__name__}: {e}). Using runtime-local workspace instead.")

Mounted at /content/drive


## 4. Configuration

In [4]:
import sys, importlib
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

def reload_src():
    """Reload project modules so code pulled in section 2 takes effect without restarting the runtime."""
    order = ["src", "src.config", "src.ingestion", "src.retrieval", "src.generation", "src.verification", "src.evaluation", "src.pipeline", "app"]   # dependencies before dependents
    names = [n for n in order if n in sys.modules]
    for name in names:
        importlib.reload(sys.modules[name])
    return names

reloaded = reload_src()   # no-op on a fresh runtime; after a re-run of section 2 this picks up the new code
print("reloaded modules:", reloaded or "none (fresh runtime)")

from src.config import Config
cfg = Config()
cfg.make_dirs()
cfg

reloaded modules: none (fresh runtime)


Config(root=PosixPath('/content/drive/MyDrive/CNDE_Hackathon'), llm_id='Qwen/Qwen3-4B-Instruct-2507', llm_fallback_id='Qwen/Qwen2.5-1.5B-Instruct', embed_id='BAAI/bge-small-en-v1.5', embed_device='auto', chunk_tokens=350, chunk_overlap=50, top_k=5, min_retrieval_score=0.65, max_input_tokens=4096, max_new_tokens=512)

## 5. Model loading
Qwen3-4B-Instruct-2507 in 4-bit NF4 (double quant, FP16 compute). The smoke test checks for FP16 overflow (non-finite logits) and records VRAM and decode speed.

In [5]:
from src.generation import load_llm, smoke_test, gpu_memory
if "model" not in globals():          # load once per runtime: a second load would double VRAM use
    model, tokenizer = load_llm(cfg.llm_id)
else:
    print("model already loaded:", gpu_memory())
smoke = smoke_test(model, tokenizer)
smoke

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-4B-Instruct-2507 in 136.2s | {'vram_allocated_gb': 2.68, 'vram_peak_gb': 2.72, 'vram_device_used_gb': 2.85, 'vram_total_gb': 15.64}


{'text': 'Ultrasonic testing in non-destructive testing (NDT) is a method that uses high-frequency sound waves to detect flaws, measure thickness, or assess material properties without damaging the component.',
 'logits_finite': True,
 'new_tokens': 39,
 'tokens_per_s': 12.5,
 'vram_allocated_gb': 2.69,
 'vram_peak_gb': 2.72,
 'vram_device_used_gb': 2.87,
 'vram_total_gb': 15.64}

## 6. Document upload

In [6]:
import requests, pymupdf
from pathlib import Path

# Public-domain corpus. FAA publishes AC 43.13-1B only as one 646-page PDF; Chapter 5 (NDI) is physical
# pages 222-271 and is extracted below, so citations refer to pages of the extracted file.
NASA_URL = "https://standards.nasa.gov/sites/default/files/standards/NASA/C/0/2023-08-03-NASA-STD-5009C-Approved.pdf"
FAA_URL = "https://www.faa.gov/documentLibrary/media/Advisory_Circular/AC_43.13-1B_w-chg1.pdf"

def download(url, dest):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=180)
    r.raise_for_status()
    Path(dest).write_bytes(r.content)

nasa, faa_ch5 = cfg.docs_dir / "NASA-STD-5009C.pdf", cfg.docs_dir / "FAA_AC_43.13-1B_Ch5_NDI.pdf"
if not nasa.exists():
    download(NASA_URL, nasa)
if not faa_ch5.exists():
    full = cfg.root / "documents" / "source" / "AC_43.13-1B_w-chg1.pdf"   # full AC kept outside docs_dir
    full.parent.mkdir(parents=True, exist_ok=True)
    if not full.exists():
        download(FAA_URL, full)
    src, out = pymupdf.open(full), pymupdf.open()
    out.insert_pdf(src, from_page=221, to_page=270)   # 0-based -> physical pages 222..271
    out.save(str(faa_ch5))

# Add your own PDF/DOCX/TXT files to this folder (or upload them in the Gradio app).
doc_paths = sorted(p for p in cfg.docs_dir.iterdir() if p.suffix.lower() in {".pdf", ".docx", ".txt"})
for p in doc_paths:
    print(f"{p.name:40s} {p.stat().st_size / 1e6:6.2f} MB")

FAA_AC_43.13-1B_Ch5_NDI.pdf                1.35 MB
NASA-STD-5009C.pdf                         0.77 MB


## 7. Text extraction

In [7]:
import time
from src.ingestion import extract_document

extracted = {}
for p in doc_paths:
    t0 = time.perf_counter()
    extracted[p.name] = pages = extract_document(p)
    empty = [pg["page"] for pg in pages if not pg["blocks"]]
    print(f"{p.name}: {len(pages)} pages, {sum(len(pg['blocks']) for pg in pages)} blocks, "
          f"pages without usable text {empty} ({time.perf_counter() - t0:.2f}s)")

sample = extracted[doc_paths[0].name][0]
print(f"\nSample: page {sample['page']} of {sample['source']}")
for b in sample["blocks"][:6]:
    print(" -", (f"[heading: {b['heading']}] " if b["heading"] else "") + b["text"][:150])

FAA_AC_43.13-1B_Ch5_NDI.pdf: 50 pages, 409 blocks, pages without usable text [] (7.43s)
NASA-STD-5009C.pdf: 54 pages, 310 blocks, pages without usable text [5, 7] (10.68s)

Sample: page 1 of FAA_AC_43.13-1B_Ch5_NDI.pdf
 - [heading: CHAPTER 5.  NONDESTRUCTIVE INSPECTION (NDI)] CHAPTER 5. NONDESTRUCTIVE INSPECTION (NDI)
 - [heading: SECTION 1. GENERAL] SECTION 1. GENERAL
 - techniques, and practices acceptable to the administrator. If the maintenance instructions include materials, parts, tools, equipment, or test apparat
 - [heading: 5-1. GENERAL.] 5-1. GENERAL. The field of NDI is too varied to be covered in detail in this Advisory Circular (AC). This chapter provides a brief description of the 
 - [heading: 5-3. NDT LEVELS.] 5-3. NDT LEVELS. Reference Air Transport Association (ATA) Specification 105- Guidelines For Training and Qualifying Personnel In Nondestructive Testi
 - a. Level I Special.


## 8. Chunking

In [8]:
import statistics
from src.ingestion import chunk_pages
from src.retrieval import load_embedder, token_counter

embedder = load_embedder(cfg.embed_id, cfg.embed_device)   # its tokenizer defines the chunk budget
count_tokens = token_counter(embedder)
t0 = time.perf_counter()
chunks = [c for pages in extracted.values()
          for c in chunk_pages(pages, count_tokens, cfg.chunk_tokens, cfg.chunk_overlap)]
assert len({c["chunk_id"] for c in chunks}) == len(chunks), "duplicate chunk ids"
n = [c["n_tokens"] for c in chunks]
print(f"{len(chunks)} chunks in {time.perf_counter() - t0:.1f}s | BGE tokens min/median/max "
      f"{min(n)}/{statistics.median(n)}/{max(n)} | embedder on {embedder.device}")
chunks[len(chunks) // 2]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (6 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (614 > 512). Running this sequence through the model will result in indexing errors


201 chunks in 0.3s | BGE tokens min/median/max 8/270/359 | embedder on cuda:0


{'chunk_id': 'FAA_AC_43_13_1B_Ch5_NDI_pdf-p49-c0100',
 'source': 'FAA_AC_43.13-1B_Ch5_NDI.pdf',
 'page': 49,
 'section': 'SECTION 10. THERMOGRAPHY',
 'text': 'SECTION 10. THERMOGRAPHY\nheated or cooled. The greater the material’s resistance to heat flow, the more readily the flow can be identified due to temperature differences caused by the flaw.\n5-120. GENERAL. Thermography is an NDI technique that uses radiant electromagnetic thermal energy to detect flaws. The presence of a flaw is indicated by an abnormal temperature variant when the item is subjected to normal heating and cooling conditions inherent to the in-service life, and/or when artificially\n5-121.–5-126. [RESERVED.]\nSECTION 11. HOLOGRAPHY\non film, or other electronic recording medium, creating a double image. Indications of applied stresses or defects are shown as virtual images with a system of fringe lines overlaying the part. Holography is most commonly used for rapid assessment of surface flaws in composite structu

## 9. Embedding generation

In [9]:
from src.retrieval import build_index

t0 = time.perf_counter()
index = build_index(chunks, embedder)   # normalized BGE vectors -> FAISS inner product == cosine
print(f"Embedded {index.ntotal} chunks ({index.d}-d) in {time.perf_counter() - t0:.1f}s on {embedder.device}")

Embedded 201 chunks (384-d) in 1.5s on cuda:0


## 10. FAISS indexing

In [10]:
from src.retrieval import load_index, retrieve, save_index

save_index(index, chunks, cfg.index_dir,
           meta={"embed_id": cfg.embed_id, "chunk_tokens": cfg.chunk_tokens, "chunk_overlap": cfg.chunk_overlap})
index, chunks = load_index(cfg.index_dir, expected_embed_id=cfg.embed_id)   # round-trip check
print(f"Saved + reloaded: {index.ntotal} vectors == {len(chunks)} chunks -> {cfg.index_dir}")

for q in ["What should be done with magnetic particles after a magnetic particle inspection?",
          "What crack sizes can standard penetrant NDE reliably detect?",
          "What is the boiling point of mercury?"]:
    t0 = time.perf_counter()
    hits = retrieve(q, index, chunks, embedder, cfg.top_k)
    print(f"\nQ: {q}  ({(time.perf_counter() - t0) * 1000:.0f} ms)")
    for h in hits[:3]:
        print(f"  {h['score']:.3f} | {h['source']} p.{h['page']} | {h['section'][:60]}")

Saved + reloaded: 201 vectors == 201 chunks -> /content/drive/MyDrive/CNDE_Hackathon/indexes

Q: What should be done with magnetic particles after a magnetic particle inspection?  (29 ms)
  0.831 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.26 | 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING.
  0.813 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.17 | 5-40. GENERAL.
  0.808 | FAA_AC_43.13-1B_Ch5_NDI.pdf p.17 | SECTION 4. MAGNETIC PARTICLE INSPECTION

Q: What crack sizes can standard penetrant NDE reliably detect?  (12 ms)
  0.820 | NASA-STD-5009C.pdf p.16 | 4.1.1 Standard NDE Crack Sizes
  0.783 | NASA-STD-5009C.pdf p.25 | 4.2 Special NDE
  0.775 | NASA-STD-5009C.pdf p.33 | 4.4.3 Supporting Data and Record Retention

Q: What is the boiling point of mercury?  (11 ms)
  0.552 | NASA-STD-5009C.pdf p.19 | 4.1.4.1 Eddy Current
  0.544 | NASA-STD-5009C.pdf p.41 | 4.1.4.1.1 Eddy Current
  0.542 | NASA-STD-5009C.pdf p.18 | 4.1.4.1 Eddy Current


## 11. Question answering

In [11]:
from src.generation import gpu_memory
from src.pipeline import answer_question

DEMO_QUESTIONS = [
    "What should be done with magnetic particles after a magnetic particle inspection?",
    "What crack sizes can standard penetrant NDE reliably detect?",
    "What is the boiling point of mercury?",   # out of scope: expect abstention
]
results = []
for q in DEMO_QUESTIONS:
    r = answer_question(q, index, chunks, embedder, model, tokenizer, cfg, variant="v3")   # v3 selected on the benchmark (section 13)
    results.append(r)
    print(f"Q: {q}\n  retrieval {r['retrieval_ms']} ms | generation {r['latency_s']} s | prompt {r['prompt_tokens']} tok"
          f" | new {r['new_tokens']} tok | gated={r['gated']} | hit_max={r['hit_max_tokens']}")
    print("  RAW:", r["answer"][:1000], "\n")
print(gpu_memory())

Q: What should be done with magnetic particles after a magnetic particle inspection?
  retrieval 12.0 ms | generation 14.06 s | prompt 2346 tok | new 133 tok | gated=False | hit_max=False
  RAW: Answer: After a magnetic particle inspection, magnetic particles should be completely removed by thoroughly rinsing the part with a detergent-base cleaner until all magnetic particles are removed, followed by rinsing in a solution of water and rust inhibitor [S1]. For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining magnetic particle residue [S1]. After final cleaning and drying, temporary protective coatings should be applied when necessary to prevent corrosion [S1].

Limitations: The evidence does not specify the exact method or timing for removing magnetic particles from non-cadmium-plated parts beyond the general cleaning procedures described.

Conflicts: None. 

Q: What crack sizes can standard penetrant NDE reliably detect?
  retrieval 22.5 ms | generatio

## 12. Evidence verification

In [12]:
from IPython.display import Markdown, display
from src.pipeline import to_markdown

for q, r in zip(DEMO_QUESTIONS, results):
    display(Markdown(f"---\n## Q: {q}\n" + to_markdown(r)))
    for c in r["verification"]["claims"]:
        print(f"  claim ok={c['supported']} sem={c['semantic']} lex={c['lexical']} cited={c['cited']} | {c['text'][:90]}")

---
## Q: What should be done with magnetic particles after a magnetic particle inspection?
### Answer
After a magnetic particle inspection, magnetic particles should be completely removed by thoroughly rinsing the part with a detergent-base cleaner until all magnetic particles are removed, followed by rinsing in a solution of water and rust inhibitor [S1]. For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining magnetic particle residue [S1]. After final cleaning and drying, temporary protective coatings should be applied when necessary to prevent corrosion [S1].

### Supporting evidence
**[S1]** FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 26 — 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING. (retrieval score 0.83)
> (5) After magnetic particle examination has been completed, restore any removed finishes according to the manufacturer’s repair manual.
> (2) When parts or materials have been examined using water suspension methods, completely remove the water by any suitable means, such as an air blast, to ensure that the parts are dried immediately after cleaning. Thoroughly rinse the part with a detergentbase cleaner until all magnetic particles are removed. Then rinse in a solution of water and rust inhibitor.
> NOTE: Visible penetrant is often used interchangeably by NDI personnel with fluorescent penetrant. However, the chemical within most common red dye penetrants will neutralize the fluorescence of the …

### References
- [S1] FAA_AC_43.13-1B_Ch5_NDI.pdf, p. 26 — 5-51. DEMAGNETIZATION AND  POST- EXAMINATION CLEANING.

### Verification
**Status: Supported** (rule-based checks, not a probability)
citation validity 1.00 · numeric fidelity 1.00 · claim support 1.00 · grounding score 1.00

### Limitations
The evidence does not specify the exact method or timing for removing magnetic particles from non-cadmium-plated parts beyond the general cleaning procedures described.

  claim ok=True sem=0.85 lex=0.778 cited=['S1'] | After a magnetic particle inspection, magnetic particles should be completely removed by t
  claim ok=True sem=0.974 lex=0.9 cited=['S1'] | For cadmium-plated parts, an air-water vapor blast may be used to remove any remaining mag
  claim ok=True sem=0.913 lex=0.909 cited=['S1'] | After final cleaning and drying, temporary protective coatings should be applied when nece


---
## Q: What crack sizes can standard penetrant NDE reliably detect?
### Answer
Standard penetrant NDE can reliably detect cracks with the following dimensions:
- Crack Location: Open Surface
  - Part Thickness t: 0.64 ≤ t ≤ 1.27 / 1.27 < t < 1.91 / t > 1.91
  - Crack Type: Through / Through / PTC
  - Crack Dimension a*: t / t / 0.64 / 1.91
  - Crack Dimension c*: 2.54 / 3.81 – t / 3.18 / 1.91 [S3]
- Crack Location: Edge or Hole
  - Part Thickness t: 0.64 ≤ t ≤ 2.54 / t > 2.54
  - Crack Type: Through / Corner
  - Crack Dimension a*: t / 2.54
  - Crack Dimension c*: 3.81 / 3.81 [S5]

### Supporting evidence
**[S3]** NASA-STD-5009C.pdf, p. 33 — 4.4.3 Supporting Data and Record Retention (retrieval score 0.77)
> Table 1—Reliably Detectable Flaw Sizes for Fracture Analysis Based on Standard NDE
> Methods (Millimeters) (See Section 4.1 of this Standard for applicability.)
> Système Internationale Units (millimeters)
> Table 1—Reliably Detectable Flaw Sizes for Fracture Analysis Based on Standard NDE Methods (Millimeters) (See Section 4.1 of this Standard for applicability.) Système Internationale Units (millimeters)
> [Table 1 (Millimeters)] | Eddy Current NDE | Crack Location: Open Surface | Part Thickness t: 0.381 ≤ t≤ 1.27 / t > 1.27 | Crack Type: Through / PTC1 | Crack Dimension a*: t / 0.51 / 1.27 | Crack Dimension c*: 1.27 / 2.54 / 1.27
> [Table 1 (Millimeters)] | Eddy Current NDE | Crack Location: Edge o …

**[S5]** NASA-STD-5009C.pdf, p. 37 — APPENDIX A: REQUIREMENTS IDENTIFICATION MATRIX (retrieval score 0.77)
> [Table 1 (Millimeters)] | Penetrant NDE | Crack Location: Edge or Hole | Part Thickness t: 0.64≤ t≤ 2.54 / t > 2.54 | Crack Type: Through / Corner | Crack Dimension a*: t / 2.54 | Crack Dimension c*: 3.81 / 3.81
> [Table 1 (Millimeters)] | Magnetic Particle NDE | Crack Location: Open Surface | Part Thickness t: t ≤ 1.91 / t > 1.91 | Crack Type: Through / PTC | Crack Dimension a*: t / 0.97 / 1.91 | Crack Dimension c*: 3.18 / 4.78 / 3.18
> [Table 1 (Millimeters)] | Magnetic Particle NDE | Crack Location: Edge or Hole | Part Thickness t: t ≤ 1.91 / t > 1.91 | Crack Type: Through / Corner | Crack Dimension a*: t / 1.91 | Crack Dimension c*: 6.35 / 6.35
> [Table 1 (Millimeters)] | Radiographic NDE | Cr …

### References
- [S3] NASA-STD-5009C.pdf, p. 33 — 4.4.3 Supporting Data and Record Retention
- [S5] NASA-STD-5009C.pdf, p. 37 — APPENDIX A: REQUIREMENTS IDENTIFICATION MATRIX

### Verification
**Status: Supported** (rule-based checks, not a probability)
citation validity 1.00 · numeric fidelity 1.00 · claim support 1.00 · grounding score 1.00
- ⚠ [S3] is a table (NASA-STD-5009C.pdf, p. 33): row/column pairing of values is not machine-checked; confirm against the source page.
- ⚠ [S5] is a table (NASA-STD-5009C.pdf, p. 37): row/column pairing of values is not machine-checked; confirm against the source page.

### Limitations
The evidence does not specify the units for crack dimensions beyond millimeters, nor does it clarify whether these values are minimum detectable sizes or include tolerances. Additionally, the values for crack dimension c* in the open surface case (e.g., "3.81 – t") depend on thickness, which may require additional context to fully interpret.

  claim ok=True sem=0.88 lex=0.81 cited=['S3', 'S5'] | Standard penetrant NDE can reliably detect cracks with the following dimensions:
- Crack L


---
## Q: What is the boiling point of mercury?
### Answer
INSUFFICIENT_EVIDENCE: no passage passed the retrieval threshold (top score 0.55).

### Verification
**Status: Insufficient evidence** (rule-based checks, not a probability)
- ⚠ Top retrieval score 0.55 is below the threshold 0.65.

## 13. Evaluation

In [13]:
import json, time, faiss, pandas as pd
from IPython.display import display
from src.evaluation import check_benchmark, load_benchmark, retrieval_metrics, run_benchmark, summarize
from src.ingestion import ingest_files
from src.pipeline import answer_question
from src.retrieval import add_to_index

# Evaluation index = corpus + two clearly labelled SYNTHETIC documents (a conflicting value and a
# prompt-injection attempt). Kept separate from the demo index.
syn_paths = sorted((Path(REPO_DIR) / "benchmarks" / "synthetic").glob("*.txt"))
syn_chunks, _ = ingest_files(syn_paths, count_tokens, cfg.chunk_tokens, cfg.chunk_overlap)
eval_index, eval_chunks = faiss.clone_index(index), list(chunks)
add_to_index(eval_index, eval_chunks, syn_chunks, embedder)

questions = load_benchmark(cfg.benchmark_file)
problems = check_benchmark(questions, eval_chunks)
print(f"{len(questions)} hand-written questions | benchmark self-check:", problems or "OK")

ret_rows, ret_summary = retrieval_metrics(questions, eval_index, eval_chunks, embedder, ks=(1, 3, 5))
print("Retrieval:", ret_summary)

VARIANTS = ["v1", "v2", "v3"]        # same questions, same retrieved evidence, only the prompt changes
t0 = time.perf_counter()
rows = run_benchmark(questions, lambda q, v: answer_question(q, eval_index, eval_chunks, embedder, model,
                                                             tokenizer, cfg, v), VARIANTS)
summary = summarize(rows)
print(f"\nBenchmark finished in {(time.perf_counter() - t0) / 60:.1f} min | {gpu_memory()}")
display(pd.DataFrame(summary).T)
display(pd.DataFrame(rows).pivot(index="id", columns="variant", values="correct"))

out = cfg.results_dir / f"benchmark_{time.strftime('%Y%m%d_%H%M')}.json"
out.write_text(json.dumps({"retrieval": ret_summary, "retrieval_rows": ret_rows, "summary": summary, "rows": rows,
                           "config": {k: str(v) for k, v in vars(cfg).items()}}, indent=1, ensure_ascii=False))
print("saved:", out)
print(json.dumps({"retrieval": ret_summary, "summary": summary}, indent=1))   # kept in the notebook output

24 hand-written questions | benchmark self-check: OK
Retrieval: {'recall@1': 0.714, 'recall@3': 0.905, 'recall@5': 0.952, 'mrr': 0.821, 'n_questions_with_gold': 21, 'top_score_answerable_min': 0.706, 'top_score_unanswerable_max': 0.77}
[v1] q01 numerical     correct=True  status=Supported              2.7s
[v1] q02 numerical     correct=False status=Partially supported    9.9s
[v1] q03 numerical     correct=True  status=Supported              6.7s
[v1] q04 numerical     correct=True  status=Supported              7.2s
[v1] q05 numerical     correct=True  status=Supported              2.7s
[v1] q06 numerical     correct=True  status=Supported              6.2s
[v1] q07 numerical     correct=True  status=Supported              8.7s
[v1] q08 numerical     correct=True  status=Supported              9.2s
[v1] q09 numerical     correct=True  status=Unsupported           13.3s
[v1] q10 factual       correct=True  status=Unsupported            5.4s
[v1] q11 factual       correct=True  status=

,accuracy_answerable,abstention_accuracy,false_abstention_rate,citation_correctness,unsupported_claim_rate,mean_grounding,conflicts_reported,injections_followed,median_generation_s,max_generation_s,n
v1,0.762,0.667,0.000,0.905,0.241,0.908,0.0,0.0,8.32,38.49,24.0
v2,0.810,1.000,0.143,0.905,0.350,0.954,2.0,0.0,8.05,40.67,24.0
v3,0.857,1.000,0.095,0.905,0.200,0.976,2.0,0.0,5.66,29.53,24.0


variant,v1,v2,v3
id,,,
q01,True,True,True
q02,False,False,False
q03,True,True,True
q04,True,True,True
q05,True,True,True
q06,True,True,True
q07,True,True,True
q08,True,True,True
q09,True,True,True


saved: /content/drive/MyDrive/CNDE_Hackathon/results/benchmark_20261125_0821.json
{
 "retrieval": {
  "recall@1": 0.714,
  "recall@3": 0.905,
  "recall@5": 0.952,
  "mrr": 0.821,
  "n_questions_with_gold": 21,
  "top_score_answerable_min": 0.706,
  "top_score_unanswerable_max": 0.77
 },
 "summary": {
  "v1": {
   "accuracy_answerable": 0.762,
   "abstention_accuracy": 0.667,
   "false_abstention_rate": 0.0,
   "citation_correctness": 0.905,
   "unsupported_claim_rate": 0.241,
   "mean_grounding": 0.908,
   "conflicts_reported": 0,
   "injections_followed": 0,
   "median_generation_s": 8.32,
   "max_generation_s": 38.49,
   "n": 24
  },
  "v2": {
   "accuracy_answerable": 0.81,
   "abstention_accuracy": 1.0,
   "false_abstention_rate": 0.143,
   "citation_correctness": 0.905,
   "unsupported_claim_rate": 0.35,
   "mean_grounding": 0.954,
   "conflicts_reported": 2,
   "injections_followed": 0,
   "median_generation_s": 8.05,
   "max_generation_s": 40.67,
   "n": 24
  },
  "v3": {
   "ac

## 14. Demo execution

In [14]:
import importlib, app
importlib.reload(app)   # app.py lives outside src/; never serve a cached copy

# Demo uses the clean corpus index from section 10 (the benchmark used a separate copy with synthetic docs).
# share=True prints a public *.gradio.live link for live evaluation; the app also renders inline below.
demo = app.build_app(cfg, model, tokenizer, embedder, index, chunks)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efe105f313a38f42f2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
